# Project Title: Student Performance Prediction

# Tools & Concepts Used:-

NumPy: arrays, math operations

Pandas: data loading, cleaning, statistics

In [1]:
import pandas as pd
import numpy as np

# Objective

Predict a student’s final score based on study hours, attendance, and previous scores.


In [2]:
df = pd.read_csv("student_performance_prediction.csv")

In [3]:
df.head(5)

,Student ID,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Parent Education Level,Passed
0,S00001,12.5,NaN,75.0,Yes,Master,Yes
1,S00002,9.3,95.3,60.6,No,High School,No
2,S00003,13.2,NaN,64.0,No,Associate,No
3,S00004,17.6,76.8,62.4,Yes,Bachelor,No
4,S00005,8.8,89.3,72.7,No,Master,No


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 7 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Student ID                                   40000 non-null  object 
 1   Study Hours per Week                         38005 non-null  float64
 2   Attendance Rate                              38008 non-null  float64
 3   Previous Grades                              38006 non-null  float64
 4   Participation in Extracurricular Activities  38000 non-null  object 
 5   Parent Education Level                       38000 non-null  object 
 6   Passed                                       38000 non-null  object 
dtypes: float64(3), object(4)
memory usage: 2.1+ MB


In [5]:
df.isnull().mean() * 100

Student ID                                     0.0000
Study Hours per Week                           4.9875
Attendance Rate                                4.9800
Previous Grades                                4.9850
Participation in Extracurricular Activities    5.0000
Parent Education Level                         5.0000
Passed                                         5.0000
dtype: float64

In [6]:
df['Attendance Rate'].fillna(df['Attendance Rate'].median(), inplace=True)

In [7]:
df.describe()

,Study Hours per Week,Attendance Rate,Previous Grades
count,38005.000000,40000.000000,38006.000000
mean,9.962744,75.277502,65.440107
std,5.031154,19.879125,16.503119
min,-12.300000,-14.300000,8.300000
25%,6.600000,62.400000,55.100000
50%,10.000000,75.300000,65.200000
75%,13.400000,87.900000,75.200000
max,32.400000,150.200000,200.000000


In [8]:
df['Passed'].value_counts(normalize=True)


Passed
Yes    0.500289
No     0.499711
Name: proportion, dtype: float64

In [9]:
from scipy.stats import ttest_ind

pass_hours = df[df['Passed']=="Yes"]['Study Hours per Week']
fail_hours = df[df['Passed']=="No"]['Study Hours per Week']

t_stat, p_value = ttest_ind(pass_hours, fail_hours)
p_value

nan

In [10]:
df['Extracurricular'] = df['Participation in Extracurricular Activities'].map({'Yes': 1, 'No': 0})

df['Passed'] = df['Passed'].map({'Yes':1, 'No':0})

print(df.columns.tolist())

cols_to_dummy = ['Parent Education Level']

# Only run if the column is still in the dataframe

if all(col in df.columns for col in cols_to_dummy):
    df = pd.get_dummies(df,columns=cols_to_dummy,drop_first=True)
else:
    print("Columns already transformed or not found.")

['Student ID', 'Study Hours per Week', 'Attendance Rate', 'Previous Grades', 'Participation in Extracurricular Activities', 'Parent Education Level', 'Passed', 'Extracurricular']


In [11]:
df.head()

,Student ID,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Passed,Extracurricular,Parent Education Level_Bachelor,Parent Education Level_Doctorate,Parent Education Level_High School,Parent Education Level_Master
0,S00001,12.5,75.3,75.0,Yes,1.0,1.0,False,False,False,True
1,S00002,9.3,95.3,60.6,No,0.0,0.0,False,False,True,False
2,S00003,13.2,75.3,64.0,No,0.0,0.0,False,False,False,False
3,S00004,17.6,76.8,62.4,Yes,0.0,1.0,True,False,False,False
4,S00005,8.8,89.3,72.7,No,0.0,0.0,False,False,False,True


In [12]:
#The numeric_only=True argument ignores string columns automatically 

df.corr(numeric_only=True)['Passed'].sort_values(ascending=False)

Passed                                1.000000
Attendance Rate                       0.009049
Parent Education Level_Doctorate      0.007488
Previous Grades                       0.003720
Extracurricular                       0.001385
Parent Education Level_High School    0.001191
Parent Education Level_Bachelor       0.001188
Parent Education Level_Master        -0.003186
Study Hours per Week                 -0.009870
Name: Passed, dtype: float64

# Model selection & evaluation (train/test split, metrics)

In [13]:
from sklearn.model_selection import train_test_split

# Drop all non-numeric columns and the target variable

X = df.drop(columns=['Passed','Student ID',
                     'Participation in Extracurricular Activities'])

# If 'Parent Education Level' is still there,you must drop it too:

if 'Parent Education Level' in X.columns:
    X = X.drop(columns=['Parent Education Level']) 
    
y = df['Passed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
print(X.dtypes)

Study Hours per Week                  float64
Attendance Rate                       float64
Previous Grades                       float64
Extracurricular                       float64
Parent Education Level_Bachelor          bool
Parent Education Level_Doctorate         bool
Parent Education Level_High School       bool
Parent Education Level_Master            bool
dtype: object


In [15]:
df.isnull().sum()

Student ID                                        0
Study Hours per Week                           1995
Attendance Rate                                   0
Previous Grades                                1994
Participation in Extracurricular Activities    2000
Passed                                         2000
Extracurricular                                2000
Parent Education Level_Bachelor                   0
Parent Education Level_Doctorate                  0
Parent Education Level_High School                0
Parent Education Level_Master                     0
dtype: int64

In [16]:
df.dropna()

,Student ID,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Passed,Extracurricular,Parent Education Level_Bachelor,Parent Education Level_Doctorate,Parent Education Level_High School,Parent Education Level_Master
0,S00001,12.5,75.3,75.0,Yes,1.0,1.0,False,False,False,True
1,S00002,9.3,95.3,60.6,No,0.0,0.0,False,False,True,False
2,S00003,13.2,75.3,64.0,No,0.0,0.0,False,False,False,False
3,S00004,17.6,76.8,62.4,Yes,0.0,1.0,True,False,False,False
4,S00005,8.8,89.3,72.7,No,0.0,0.0,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...
39994,S39995,5.1,92.1,46.1,Yes,0.0,1.0,False,True,False,False
39995,S39996,15.6,93.8,51.4,Yes,0.0,1.0,False,False,False,True
39996,S39997,11.3,66.4,64.2,No,1.0,0.0,False,True,False,False
39997,S39998,13.1,65.6,38.1,No,0.0,0.0,True,False,False,False


In [17]:
X = df.drop(columns=['Passed','Student ID',
                     'Participation in Extracurricular Activities','Parent Education Level'],errors='ignore')
y = df['Passed']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
# List of columns that are are strings or not useful for the model 

cols_to_drop = ['Student ID',
                     'Participation in Extracurricular Activities','Parent Education Level']

# Drop them safely

df_final = df.drop(columns=[col 
                            for col in cols_to_drop if col in df.columns])

In [19]:
# Fill missing numerical values with the median

df_final = df_final.fillna(df_final.median())

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#Define X(features) and y (target)

X = df_final.drop(columns=['Passed'])
y = df_final['Passed']

#Split the data

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

#Train the model

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

#Checks the results

y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test,y_pred): .2%}")

Model Accuracy:  52.61%


In [21]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.526125
[[   3 3791]
 [   0 4206]]
              precision    recall  f1-score   support

         0.0       1.00      0.00      0.00      3794
         1.0       0.53      1.00      0.69      4206

    accuracy                           0.53      8000
   macro avg       0.76      0.50      0.35      8000
weighted avg       0.75      0.53      0.36      8000



In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [23]:
import pandas as pd

importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

Attendance Rate                       0.343579
Previous Grades                       0.338486
Study Hours per Week                  0.299633
Extracurricular                       0.005937
Parent Education Level_Master         0.003174
Parent Education Level_Doctorate      0.003145
Parent Education Level_Bachelor       0.003076
Parent Education Level_High School    0.002969
dtype: float64